In [ ]:
!nvidia-smi

Mon Sep  7 18:29:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!unzip -o -q legal-qa-sft.zip -d /content/

In [ ]:
%cd /content/legal-qa-sft

/content/legal-qa-sft


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!python -m py_compile data/make_dataset.py eval/metrics.py eval/robustness.py deploy/benchmark_latency.py deploy/gradio_app.py && echo '===== 语法自测通过 ====='


===== 语法自测通过 =====


In [ ]:

# 安装依赖（llamafactory 安装约 3-5 分钟，可能出现依赖告警，可忽略）
%pip install -q llamafactory rouge-chinese sacrebleu jieba bert-score
import llamafactory
print('llamafactory', llamafactory.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 20.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 26.2.1 requires cuda-toolkit[nvcc,nvrtc]==12.*, but you have cuda-toolkit 13.0.3.0 which is incompatible.
hf-gradio 0.4.1 requires gradio-client<3.0,>=2.0, but you have gradio-client 1.14.0 which is incompatible.
bigframes 2.42.0 requires rich<14,>=12.4.4, but you have rich 15.0.0 which is incompatible.
pyiceberg 0.11.1 requires rich<15.0.0,>=10.11.0, but you have rich 15.0.0 which is incompatible.
llamafactory 0.9.5


In [ ]:
# 阶段 1：数据构建（下载 DISC-Law-SFT 约 300MB → 清洗 → 5000/300/300 切分）
!python data/make_dataset.py --train 5000 --val 300 --test 300

[1/4] 下载 ShengbinYue/DISC-Law-SFT/DISC-Law-SFT-Pair.jsonl ...

DISC-Law-SFT-Pair.jsonl: downloading bytes:  28% 97.5M/347M [00:02<00:03, 74.0MB/s, 8.46MB/s  ]
DISC-Law-SFT-Pair.jsonl: reconstructing file:  58% 201M/347M [00:02<00:01, 82.6MB/s]
DISC-Law-SFT-Pair.jsonl: downloading bytes:  37% 127M/347M [00:02<00:02, 81.9MB/s, 10.3MB/s  ] 
DISC-Law-SFT-Pair.jsonl: downloading bytes: 100% 127M/127M [00:05<00:00, 22.9MB/s, 11.6MB/s  ]
DISC-Law-SFT-Pair.jsonl: reconstructing file: 100% 347M/347M [00:05<00:00, 62.7MB/s, 26.1MB/s  ]
      原始样本：166758
[2/4] 清洗（长度过滤 + 问题级去重）...
      清洗后：64361（含法条引用 11503，17.9%）
[3/4] 分层抽样与切分 ...
[4/4] 写出数据文件 ...
{
  "raw": 166758,
  "cleaned": 64361,
  "train": 5000,
  "val": 300,
  "test": 300,
  "test_citation_rate": 1.0,
  "train_top_categories": {
    "unknown": 5000
  },
  "seed": 42
}


In [ ]:
# 阶段 2：冒烟训练（0.5B + 500 样本，约 10 分钟）
!llamafactory-cli train configs/smoke_0.5b.yaml

/usr/local/lib/python3.13/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.13/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.13/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[INFO|2026-09-07 18:40:56] llamafactory.hparams.parser:523 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.float16
config.json: 100% 659/659 [00:00<00:00, 2.87MB/s]
[INFO|configuration_utils.py:771] 2026-09-07 18:40:57,091 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B-Instruc

In [ ]:
# 阶段 2：冒烟推理 + 评测（验证 训练→推理→评测 全链路）
!llamafactory-cli train configs/infer_smoke.yaml


[INFO|2026-09-07 19:15:52] llamafactory.hparams.parser:523 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.float16
[INFO|configuration_utils.py:771] 2026-09-07 19:15:52,814 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B-Instruct/snapshots/7ae557604adf67be50417f59c2c2f167def9a775/config.json
[INFO|configuration_utils.py:847] 2026-09-07 19:15:52,822 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "fu

In [ ]:
!python eval/metrics.py --pred results/smoke-0.5b/generated_predictions.jsonl

样本数：30，空预测：0
{
  "n": 30,
  "rouge_l": 0.3491,
  "bleu_char": 26.59,
  "citation": {
    "n_ref_with_citation": 30,
    "format_rate": 0.7667,
    "precision": 0.4167,
    "recall": 0.4,
    "exact_match": 0.3333
  }
}


In [ ]:

# 阶段 3：正式训练 Qwen2.5-1.5B（约 30-40 分钟）
# 关注每个 epoch 的 eval_loss：持续上升说明过拟合，可把 num_train_epochs 降为 2.0
!llamafactory-cli train configs/train_1.5b.yaml

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[INFO|2026-09-07 20:23:21] llamafactory.hparams.parser:523 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.float16
[INFO|configuration_utils.py:771] 2026-09-07 20:23:22,103 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json
[INFO|configuration_utils.py:847] 2026-09-07 20:23:22,108 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attenti

In [ ]:
ls -la saves/qwen2.5-1.5b-lora/adapter_model.safetensors

-rw-rw-rw- 1 root root 73911112 Sep  8 00:01 saves/qwen2.5-1.5b-lora/adapter_model.safetensors


In [ ]:
!llamafactory-cli train configs/infer_1.5b.yaml

[INFO|2026-09-08 01:21:59] llamafactory.hparams.parser:523 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.float16
config.json: 100% 660/660 [00:00<00:00, 2.89MB/s]
[INFO|configuration_utils.py:771] 2026-09-08 01:21:59,819 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json
[INFO|configuration_utils.py:847] 2026-09-08 01:21:59,826 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",


In [ ]:
!llamafactory-cli train configs/infer_baseline_1.5b.yaml

[INFO|2026-09-08 02:08:41] llamafactory.hparams.parser:523 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.float16
[INFO|configuration_utils.py:771] 2026-09-08 02:08:41,568 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json
[INFO|configuration_utils.py:847] 2026-09-08 02:08:41,577 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "f

In [ ]:
# 阶段 4：评测对比（BERTScore 首次会下载中文 BERT 约 400MB）
!python eval/metrics.py --pred results/ft-1.5b/generated_predictions.jsonl --bertscore --out eval_results/ft-1.5b.json
!python eval/metrics.py --pred results/base-1.5b/generated_predictions.jsonl --bertscore --out eval_results/base-1.5b.json

import json
ft = json.load(open('eval_results/ft-1.5b.json', encoding='utf-8'))
base = json.load(open('eval_results/base-1.5b.json', encoding='utf-8'))
print(f"{'指标':<18}{'基座 zero-shot':>14}{'微调后':>12}")
print(f"{'ROUGE-L':<20}{base['rouge_l']:>12.4f}{ft['rouge_l']:>12.4f}")
print(f"{'BLEU (char)':<20}{base['bleu_char']:>12.2f}{ft['bleu_char']:>12.2f}")
print(f"{'BERTScore F1':<20}{base.get('bertscore_f1', 0):>12.4f}{ft.get('bertscore_f1', 0):>12.4f}")
print(f"{'引用格式合规率':<19}{base['citation']['format_rate']:>12.2%}{ft['citation']['format_rate']:>12.2%}")
print(f"{'引用事实正确率':<19}{base['citation']['precision']:>12.2%}{ft['citation']['precision']:>12.2%}")
print(f"{'引用完全一致率':<19}{base['citation']['exact_match']:>12.2%}{ft['citation']['exact_match']:>12.2%}")


样本数：300，空预测：0
Loading weights: 100% 199/199 [00:00<00:00, 4865.50it/s]
[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
{
  "n": 300,
  "rouge_l": 0.4939,
  "bleu_char": 47.43,
  "citation": {
    "n_ref_with_citation": 300,
    "format_rate": 0.97,
    "precision": 0.675,
    "recall": 0.6494,
    "exact_ma

In [ ]:
# 阶段 4.5：鲁棒性测试（60 条 = 20 × 3 扰动，约 5 分钟）
!python eval/robustness.py --n 20


生成 60 条鲁棒性样本（20 条 × 3 种扰动）
下一步：llamafactory-cli train configs/infer_robustness.yaml，再运行 eval/metrics.py 与干净测试集对比


In [ ]:
!llamafactory-cli train configs/infer_robustness.yaml

[INFO|2026-09-08 02:49:46] llamafactory.hparams.parser:523 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.float16
[INFO|configuration_utils.py:771] 2026-09-08 02:49:46,187 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json
[INFO|configuration_utils.py:847] 2026-09-08 02:49:46,191 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "f

In [ ]:
!python eval/metrics.py --pred results/ft-1.5b-robust/generated_predictions.jsonl --out eval_results/ft-1.5b-robust.json
print('对比干净测试集的引用格式合规率，观察三类扰动下的退化幅度')

样本数：300，空预测：0
{
  "n": 300,
  "rouge_l": 0.4939,
  "bleu_char": 47.43,
  "citation": {
    "n_ref_with_citation": 300,
    "format_rate": 0.97,
    "precision": 0.675,
    "recall": 0.6494,
    "exact_match": 0.6033
  }
}
已写入 eval_results/ft-1.5b-robust.json
对比干净测试集的引用格式合规率，观察三类扰动下的退化幅度


In [ ]:
# 阶段 5：合并 LoRA → 导出 GGUF（Q8_0 约 1.6GB）
!llamafactory-cli export configs/merge_1.5b.yaml

In [ ]:
%cd /content/legal-qa-sft
!python /content/llama.cpp/convert_hf_to_gguf.py \
    /content/legal-qa-sft/models/qwen2.5-1.5b-legal-merged \
    --outtype q8_0 \
    --outfile /content/legal-qa-sft/models/qwen2.5-1.5b-legal-merged.Q8_0.gguf

/content/legal-qa-sft
INFO:hf-to-gguf:Loading model: qwen2.5-1.5b-legal-merged
INFO:numexpr.utils:NumExpr defaulting to 2 threads.
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00002.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00002.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.bfloat16 --> Q8_0, shape = {1536, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.bfloat16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.bfloat16 --> Q8_0, shape = {8960, 1536}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.bfloat16 --> Q8_0, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.bfloat16 --> Q8_0, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_norm.weig

In [ ]:
!CMAKE_ARGS='-DGGML_CUDA=on' pip install -q llama-cpp-python



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 12.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.1 MB/s eta 0:00:00
加载 models/qwen2.5-1.5b-legal-q8_0.gguf ...
Traceback (most recent call last):
  File "/content/legal-qa-sft/deploy/benchmark_latency.py", line 84, in <module>
    main()
    ~~~~^^
  File "/content/legal-qa-sft/deploy/benchmark_latency.py", line 42, in main
    llm = Llama(model_path=args.model, n_ctx=2048,
                n_gpu_layers=args.n_gpu_layers, verbose=False)
  File "/usr/local/lib/python3.13/dist-packages/llama_cpp/llama.py", line 384, in __init__
    raise ValueError(f"Model path does not exist: {model_path}")
ValueError: Model path does not exist: models/qwen2.5-1.5b-legal-q8_0.gguf


In [ ]:
!python deploy/benchmark_latency.py --model models/qwen2.5-1.5b-legal-merged.Q8_0.gguf -n 30 --n-gpu-layers -1

加载 models/qwen2.5-1.5b-legal-merged.Q8_0.gguf ...
[1/30] ttft=0.25s  e2e=2.80s  96.9 tok/s
[2/30] ttft=0.07s  e2e=2.39s  98.1 tok/s
[3/30] ttft=0.07s  e2e=2.75s  95.0 tok/s
[4/30] ttft=0.05s  e2e=2.25s  98.8 tok/s
[5/30] ttft=0.05s  e2e=1.50s  100.4 tok/s
[6/30] ttft=0.06s  e2e=2.65s  97.9 tok/s
[7/30] ttft=0.08s  e2e=2.66s  98.3 tok/s
[8/30] ttft=0.09s  e2e=2.31s  97.5 tok/s
[9/30] ttft=0.05s  e2e=1.80s  96.6 tok/s
[10/30] ttft=0.07s  e2e=2.40s  96.9 tok/s
[11/30] ttft=0.05s  e2e=1.23s  101.1 tok/s
[12/30] ttft=0.08s  e2e=2.68s  98.1 tok/s
[13/30] ttft=0.08s  e2e=1.97s  97.1 tok/s
[14/30] ttft=0.05s  e2e=1.61s  99.7 tok/s
[15/30] ttft=0.05s  e2e=2.33s  95.8 tok/s
[16/30] ttft=0.06s  e2e=1.92s  94.4 tok/s
[17/30] ttft=0.06s  e2e=2.70s  96.7 tok/s
[18/30] ttft=0.05s  e2e=2.65s  97.6 tok/s
[19/30] ttft=0.07s  e2e=1.62s  97.7 tok/s
[20/30] ttft=0.07s  e2e=2.70s  96.8 tok/s
[21/30] ttft=0.09s  e2e=2.78s  95.0 tok/s
[22/30] ttft=0.05s  e2e=2.72s  95.2 tok/s
[23/30] ttft=0.07s  e2e=2.72s  96

In [ ]:
# 收尾：全部产物保存到 Google Drive（Colab 会话会断，务必执行）
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/legal-qa-sft-output
!cp -r saves /content/drive/MyDrive/legal-qa-sft-output/ 2>/dev/null || true
!cp -r results /content/drive/MyDrive/legal-qa-sft-output/ 2>/dev/null || true
!cp -r eval_results /content/drive/MyDrive/legal-qa-sft-output/ 2>/dev/null || true
!cp models/*.gguf /content/drive/MyDrive/legal-qa-sft-output/ 2>/dev/null || true
!cp data/data_stats.json /content/drive/MyDrive/legal-qa-sft-output/ 2>/dev/null || true
print('已保存到 Drive: legal-qa-sft-output/')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
已保存到 Drive: legal-qa-sft-output/


In [ ]:
!cp -r /content/legal-qa-sft /content/drive/MyDrive/

In [ ]:
!zip -j predictions.zip results/ft-1.5b/generated_predictions.jsonl results/base-1.5b/generated_predictions.jsonl
from google.colab import files; files.download('predictions.zip')